# NYC AirBNB

## Part 1: Data Cleaning

Note: I modified the Original Cleanup for better models. I also removed graphs since since already done in the previous project

In [83]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [90]:
df = pd.read_csv('AB_NYC_2019.csv')

In [91]:
df.isna().sum()

id                                    0
name                                 16
host_id                               0
host_name                            21
neighbourhood_group                   0
neighbourhood                         0
latitude                              0
longitude                             0
room_type                             0
price                                 0
minimum_nights                        0
number_of_reviews                     0
last_review                       10052
reviews_per_month                 10052
calculated_host_listings_count        0
availability_365                      0
dtype: int64

In [92]:
# Converting 'last_review' to datetime, coercing errors to NaT
df["last_review"] = pd.to_datetime(df["last_review"], errors="coerce")

#Filling null values in 'reviews_per_month' with 0
df['reviews_per_month'] = df['reviews_per_month'].fillna(0)

#Dropped these columns because they are not useful for data modelling
df.drop(columns=['id', 'host_id', 'name', 'host_name', 'last_review'], inplace=True)

#Removed outliers in 'price' column
df = df[df["price"] <= 500]

In [93]:
df.isna().sum()

neighbourhood_group               0
neighbourhood                     0
latitude                          0
longitude                         0
room_type                         0
price                             0
minimum_nights                    0
number_of_reviews                 0
reviews_per_month                 0
calculated_host_listings_count    0
availability_365                  0
dtype: int64

In [95]:
df

,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,reviews_per_month,calculated_host_listings_count,availability_365
0,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,0.21,6,365
1,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,0.38,2,355
2,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,0,0.00,1,365
3,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,89,1,270,4.64,1,194
4,Manhattan,East Harlem,40.79851,-73.94399,Entire home/apt,80,10,9,0.10,1,0
...,...,...,...,...,...,...,...,...,...,...,...
48890,Brooklyn,Bedford-Stuyvesant,40.67853,-73.94995,Private room,70,2,0,0.00,2,9
48891,Brooklyn,Bushwick,40.70184,-73.93317,Private room,40,4,0,0.00,2,36
48892,Manhattan,Harlem,40.81475,-73.94867,Entire home/apt,115,10,0,0.00,1,27
48893,Manhattan,Hell's Kitchen,40.75751,-73.99112,Shared room,55,1,0,0.00,6,2


## Part 2: Model

In [96]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# The target is price, so we separate it from the features
X = df.drop('price', axis=1)
y = df['price']

# Convert categorical columns to numbers for sklearn models
X = pd.get_dummies(X, drop_first=True)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=20)

# Helper function to evaluate the models
def evaluate_model(model_name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(model_name)
    print('MAE:', mae)
    print('RMSE:', rmse)
    print('R2 Score:', r2)
    print()
    return {'Model': model_name, 'MAE': mae, 'RMSE': rmse, 'R2 Score': r2}

### Model Training and Evaluation

In [97]:
# Model 1: Random Forest Regressor
rf = RandomForestRegressor(n_estimators=100, random_state=20)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_result = evaluate_model("Random Forest Regressor", y_test, rf_pred)

# Model 2: Gradient Boosting Regressor
gb = GradientBoostingRegressor(random_state=20)
gb.fit(X_train, y_train)
gb_pred = gb.predict(X_test)
gb_result = evaluate_model("Gradient Boosting Regressor", y_test, gb_pred)

# Compare the models
results = pd.DataFrame([rf_result, gb_result]).sort_values("RMSE").reset_index(drop=True)
results

Random Forest Regressor
MAE: 40.37548864586236
RMSE: 61.330934776955665
R2 Score: 0.5112998485222857

Gradient Boosting Regressor
MAE: 41.32803032725722
RMSE: 62.236441348633356
R2 Score: 0.49676271642335557



,Model,MAE,RMSE,R2 Score
0,Random Forest Regressor,40.375489,61.330935,0.511300
1,Gradient Boosting Regressor,41.328030,62.236441,0.496763


### Suggestions to Improve the Models

Random Forest performed slightly better here because it has a lower MAE and RMSE, plus a slightly higher R2 score than Gradient Boosting.

To improve both models, I could tune `n_estimators`, `max_depth`, and `min_samples_split`.

If I wanted a simpler model, I could also reduce high-cardinality features like `neighbourhood` or group rare neighbourhoods together.

A log transform on `price` may also help because the target is still right-skewed.